In [ ]:
!pip install matplotlib
!pip install Qiskit
!pip install qiskit_aer
!pip install qiskit qiskit-aer qiskit-ibm-runtime numpy
!pip install mthree
!pip install qiskit_aer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import mthree
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.circuit.library import real_amplitudes
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.quantum_info import Statevector, state_fidelity
from scipy.optimize import minimize
from qiskit_aer.noise import NoiseModel
import heapq
from collections import Counter


#Main Solution

In [ ]:


# --- 1. SETUP DNA DATA & MAPPING ---
full_seq = "ATGCGTACGTTAGCGTACGATCGTAGCTAGCTTGACGATCGTACGTTAGC"
half_len = len(full_seq) // 2
sequences = [full_seq[:half_len], full_seq[half_len:]]

base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
reverse_map = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}

# 5 index qubits (for 25 bases) + 2 data qubits = 7 qubits total
num_qubits = 7
backend = FakeSherbrooke()
mit = mthree.M3Mitigation(backend)

def solve_fragment(dna_fragment, fragment_id):
    print(f"\n--- Processing Fragment {fragment_id} ({len(dna_fragment)} bases) ---")

    # Target State Construction
    target_sv_array = np.zeros(2**num_qubits, dtype=complex)
    for i, base in enumerate(dna_fragment):
        idx = (i << 2) + base_map[base]
        target_sv_array[idx] = 1.0 / np.sqrt(len(dna_fragment))
    target_sv = Statevector(target_sv_array)

    # Ansatz Definition
    ansatz = real_amplitudes(num_qubits, entanglement='linear', reps=6)

    # Optimization
    def cost_func(params):
        assigned_ansatz = ansatz.assign_parameters(params)
        return 1 - state_fidelity(target_sv, Statevector(assigned_ansatz))

    print(f"Optimizing Fragment {fragment_id}...")
    res = minimize(cost_func, np.random.rand(ansatz.num_parameters),
                   method='COBYLA', options={'maxiter': 800})

    # Hardware Execution
    final_ansatz = ansatz.assign_parameters(res.x)
    noisy_qc = final_ansatz.copy()
    noisy_qc.measure_all()

    transpiled_qc = transpile(noisy_qc, backend, optimization_level=3)
    print(f"Transpiled Depth: {transpiled_qc.depth()}")

    counts = backend.run(transpiled_qc, shots=20000).result().get_counts()

    # Mitigation
    mapping = mthree.utils.final_measurement_mapping(transpiled_qc)
    mit.cals_from_system(mapping)
    quasi_probs = mit.apply_correction(counts, mapping)
    clean_probs = quasi_probs.nearest_probability_distribution()

    # Calculate Fragment Fidelity
    ideal_probs = Statevector(final_ansatz).probabilities()
    mitigated_vec = np.zeros(2**num_qubits)
    for b, p in clean_probs.items(): mitigated_vec[int(b, 2)] = p
    fid = np.sum(np.sqrt(ideal_probs * mitigated_vec))**2

    # Decode Fragment
    recovered_frag = ""
    for i in range(len(dna_fragment)):
        votes = {'A': 0, 'C': 0, 'G': 0, 'T': 0}
        for bitstring, val in clean_probs.items():
            val_int = int(bitstring, 2)
            if (val_int >> 2) == i:
                votes[reverse_map[val_int & 0x3]] += val
        recovered_frag += max(votes, key=votes.get)

    return recovered_frag, fid

# Run both fragments
rec_1, fid_1 = solve_fragment(sequences[0], 1)
rec_2, fid_2 = solve_fragment(sequences[1], 2)

# Combine
final_recovered = rec_1 + rec_2
total_accuracy = sum(1 for a, b in zip(full_seq, final_recovered) if a == b)

# --- 2. FINAL OUTPUT ---
print(f"\n" + "="*50)
print(f"Final Results (Stitched Fragments)")
print(f"Fragment 1 Fidelity: {fid_1:.4f}")
print(f"Fragment 2 Fidelity: {fid_2:.4f}")
print(f"Total Match:         {full_seq == final_recovered}")
print(f"Accuracy Score:      {total_accuracy}/50")
print(f"Original:  {full_seq}")
print(f"Recovered: {final_recovered}")
print(f"="*50)

#Basic Encoding Approach
While trying to reduce the number of qubits, we improvised and made a 2-qubit code. The issue with this is that it basically acts as a "very costly classical computer". It just uses X gates, bitwise.

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# --- 1. Problem Setup ---
dna_seq = 'ATGCGTACGTTAGCGTACGATCGTAGCTAGCTTGACGATCGTACGTTAGC'
base_map = {'A': '00', 'C': '01', 'G': '10', 'T': '11'}

# CORRECTED: Build the ideal Qiskit string by prepending each 2-bit block intact
ideal_bitstring_qiskit_format = ""
for base in dna_seq:
    ideal_bitstring_qiskit_format = base_map[base] + ideal_bitstring_qiskit_format

# Print it just to be safe!
print("Target String:", ideal_bitstring_qiskit_format)
# --- 2. Dynamic Circuit Construction (The Loop) ---
# We use exactly 2 qubits for encoding, and 100 classical bits for memory
qr = QuantumRegister(2, 'q')
cr = ClassicalRegister(len(dna_seq)*2, 'c')
qc = QuantumCircuit(qr, cr)


#WE ARE REUSING QUBITS here
for i, base in enumerate(dna_seq):
    val = base_map[base]

    # Reset the qubits to |00> before encoding the next base
    if i > 0:
        qc.reset(qr) #JUST to make sure all except 0th qubit is reset to ground state (|0>)

    # Encode the base using minimal X gates
    if val[1] == '1': # Right bit maps to Qubit 0
        qc.x(qr[0])
    if val[0] == '1': # Left bit maps to Qubit 1
        qc.x(qr[1])

    # Measure the quantum state directly into the classical array
    qc.measure(qr[0], cr[i*2])
    qc.measure(qr[1], cr[i*2 + 1])

    # --- 3. Hardware Compilation ---
noisy_backend = FakeSherbrooke()

print("Compiling Dynamic Circuit for FakeSherbrooke...")
# We transpile to the backend to get the real metrics
transpiled_qc = transpile(qc, backend=noisy_backend, optimization_level=3)

# --- 4. Extracting Judging Metrics ---
depth = transpiled_qc.depth()
gate_counts = transpiled_qc.count_ops()
two_qubit_count = gate_counts.get('ecr', 0) + gate_counts.get('cx', 0) #ECR and CNOT Gate combined
swap_count = gate_counts.get('swap', 0)

print("\n--- JUDGING CRITERIA METRICS ---")
print(f"1. Qubits Used: {transpiled_qc.num_qubits} (But only 2 are active/measured!)")
print(f"2. Circuit Depth: {depth}")
print(f"3. Two-Qubit Gate Count (ECR): {two_qubit_count}")
print(f"4. SWAP Gate Count: {swap_count}")

# --- 5. Simulation & Bitwise Fidelity Calculation ---
noise_model = NoiseModel.from_backend(noisy_backend)
noisy_sim = AerSimulator(noise_model=noise_model)

shots = 2**12
print("\nRunning noisy simulation with mid-circuit resets...")
result = noisy_sim.run(transpiled_qc, shots=shots).result()
counts = result.get_counts()

# Calculate Average Bitwise Fidelity
total_bits_measured = 100 * shots
correct_bits = 0

# Compare every measured bitstring against the ideal string
for bitstring, count in counts.items():
    # Count how many individual bits match the target
    for i in range(100):
        if bitstring[i] == ideal_bitstring_qiskit_format[i]:
            correct_bits += count

bitwise_fidelity = correct_bits / total_bits_measured

print(f"5. Average Bitwise Fidelity: {bitwise_fidelity:.4f} ({(bitwise_fidelity*100):.2f}%)")

#Huffman Encoding Approach
Huffman Encoding is a classical algorithm to assign more important/frequent elements shorter numbers. We tried to apply this in a more "quantum" manner.
The catch is Huffman coding does no good than 2 bit-encoding (in terms of number of qubits taken) when the A, T, G, C characters are equally distributed

In [ ]:

# -----------------------------
# Huffman Coding
# -----------------------------

class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_tree(sequence):
    freq = Counter(sequence)

    heap = []
    for char, f in freq.items():
        heapq.heappush(heap, Node(char, f))

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        merged = Node(None, left.freq + right.freq)
        merged.left = left
        merged.right = right

        heapq.heappush(heap, merged)

    return heap[0]


def generate_codes(node, prefix="", codebook=None):
    if codebook is None:
        codebook = {}

    if node.char is not None:
        codebook[node.char] = prefix
        return codebook

    generate_codes(node.left, prefix + "0", codebook)
    generate_codes(node.right, prefix + "1", codebook)

    return codebook


def huffman_encode(sequence):
    tree = build_huffman_tree(sequence)
    codes = generate_codes(tree)

    encoded = "".join(codes[c] for c in sequence)
    avg_bits = len(encoded) / len(sequence)

    return codes, encoded, avg_bits


# -----------------------------
# Input DNA
# -----------------------------

sequence = "ATGCGTACGTTAGCGTACGATCGTAGCTAGCTTGACGATCGTACGTTAGC"

codes, encoded_seq, avg_bits = huffman_encode(sequence)

print("Huffman Codes:")
for k, v in codes.items():
    print(k, ":", v)

print("\nAverage bits per base:", avg_bits)
print("Encoded bitstring length:", len(encoded_seq))


# -----------------------------
# Circuit Metrics
# -----------------------------

n = len(encoded_seq)

qc_full = QuantumCircuit(n)

for i in range(n):
    if encoded_seq[i] == '1':
        qc_full.x(i)

depth = qc_full.depth()
ops = qc_full.count_ops()

two_qubit_gates = 0
swap_gates = ops.get("swap", 0)

for gate, count in ops.items():
    if gate in ["cx", "cz", "swap", "iswap", "ecr"]:
        two_qubit_gates += count

print("\nCircuit Metrics")
print("------------------")
print("Qubits used:", n)
print("Circuit depth:", depth)
print("2-qubit gate count:", two_qubit_gates)
print("SWAP gate count:", swap_gates)
print("Gate counts:", ops)


# -----------------------------
# Truncate for simulation
# -----------------------------

max_qubits = 20 #Originally 15
encoded_sim = encoded_seq[:max_qubits]

qc = QuantumCircuit(max_qubits, max_qubits)

for i in range(max_qubits):
    if encoded_sim[i] == '1':
        qc.x(i)

qc.measure(range(max_qubits), range(max_qubits))


# -----------------------------
# Ideal Simulation
# -----------------------------

shots = 2**12

ideal_sim = AerSimulator()

ideal_result = ideal_sim.run(qc, shots=shots).result()
ideal_counts = ideal_result.get_counts()


# -----------------------------
# Noisy Simulation
# -----------------------------

backend = FakeSherbrooke()

noisy_sim = AerSimulator.from_backend(backend)

transpiled = transpile(qc, backend)

noisy_result = noisy_sim.run(transpiled, shots=shots).result()
noisy_counts = noisy_result.get_counts()


# -----------------------------
# Convert counts to probability
# -----------------------------

def counts_to_prob(counts, shots):
    prob = {}
    for k, v in counts.items():
        prob[k] = v / shots
    return prob


P = counts_to_prob(ideal_counts, shots)
Q = counts_to_prob(noisy_counts, shots)


# -----------------------------
# Classical Fidelity
# -----------------------------

keys = set(P.keys()).union(Q.keys())

fid = sum(np.sqrt(P.get(k,0)*Q.get(k,0)) for k in keys)**2

print("\nFidelity:", fid)